In [1]:
import os, re, time, random, hashlib, requests
from urllib.parse import urlparse
from dotenv import load_dotenv
import trafilatura
import numpy as np
import chromadb
from groq import Groq


from google import genai
from google.genai import types
from google.genai.errors import ClientError
from sentence_transformers import SentenceTransformer
from concurrent.futures import ThreadPoolExecutor, as_completed

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") #gemini for embeddings (all)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")   # Custom Search JSON API key
BRAVE_API_KEY = os.getenv("BRAVE_API_KEY")   # Custom Search JSON API key
GOOGLE_CSE_ID  = os.getenv("GOOGLE_CSE_ID")    

GROQ_API_KEY = os.getenv("GROQ_API_KEY") # Groq for answer generation
if not GROQ_API_KEY:
    raise RuntimeError("Missing GROQ_API_KEY in .env")

groq_client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")


if not GEMINI_API_KEY:
    raise RuntimeError("Missing GEMINI_API_KEY in .env")
if not GOOGLE_CSE_ID:
    raise RuntimeError("Missing GOOGLE_CSE_ID in .env (cx)")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_LLM = os.getenv("MODEL_LLM", "gemini-2.5-flash")
MODEL_EMB = os.getenv("MODEL_EMB", "gemini-embedding-001")

EMBED_DIM = 768

TOPK = 50
TOPN_WEB = 60

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 150
MAX_CHUNKS_PER_PAGE = 12

MAX_NEW_CHUNKS_TO_EMBED_PER_RUN = 20
BATCH_SIZE = 8

SEARCH_CACHE = {}
FETCH_CACHE = {}

st_model = SentenceTransformer("all-MiniLM-L6-v2")  # 384 dims

chroma = chromadb.PersistentClient(path="./chroma_db")
col = chroma.get_or_create_collection(name="medical_rag_hybrid")

#throttling for gemini API calls
MIN_SECONDS_BETWEEN_GEMINI_CALLS = 1.2
MAX_RETRIES = 6
BACKOFF_BASE = 1.6
BACKOFF_JITTER = 0.35
_last_gemini_call_ts = 0.0


deferral_phrases = (
    "not available for this region",
    "no data for",
    "would require regional data",
    "would require national data",
    "insufficient regional data",
    "not reported at this level",
    "only available at",
    "data is limited to",
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [2]:
def _throttle_gemini():
    global _last_gemini_call_ts
    now = time.time()
    wait = MIN_SECONDS_BETWEEN_GEMINI_CALLS - (now - _last_gemini_call_ts)
    if wait > 0:
        time.sleep(wait)
    _last_gemini_call_ts = time.time()

def _is_rate_limit(err):
    s = str(err).lower()
    return ("429" in s) or ("resource_exhausted" in s) or ("rate limit" in s) or ("rpm" in s)

def gemini_call_with_retry(fn):
    for attempt in range(MAX_RETRIES + 1):
        try:
            _throttle_gemini()
            return fn()
        except ClientError as e:
            if (not _is_rate_limit(e)) or attempt == MAX_RETRIES:
                raise
            sleep_s = (BACKOFF_BASE ** attempt)
            sleep_s *= (1.0 + random.uniform(-BACKOFF_JITTER, BACKOFF_JITTER))
            time.sleep(max(0.5, sleep_s))

def safe_embed_content(model, contents, config):
    return gemini_call_with_retry(lambda: client.models.embed_content(model=model, contents=contents, config=config))

def safe_generate_content(model, contents):
    """
    Groq-only for answer generation.
    Keeps the same signature so your pipeline stays unchanged.
    """
    #`contents` is a single prompt string; sending as user content.
    # system instruction to enforce "context-only" + citations.
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": "Answer using ONLY the provided context. Use citations like [1], [2]. If you cannot answer, say what is missing."
            },
            {"role": "user", "content": contents},
        ],
        temperature=0.2,
    )

    #gemini response shape (resp.text)
    class _Resp:
        def __init__(self, text):
            self.text = text

    text = (resp.choices[0].message.content or "").strip()
    return _Resp(text)

#helper funcs
def _norm(v):
    v = np.array(v, dtype=np.float32)
    n = np.linalg.norm(v)
    return (v / n).tolist() if n > 0 else v.tolist()

def domain_of(url):
    try:
        return urlparse(url).netloc.lower()
    except:
        return ""

def chunk_text(text):
    text = re.sub(r"\s+", " ", text).strip()
    out = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    i = 0
    while i < len(text):
        c = text[i:i+CHUNK_SIZE].strip()
        if c:
            out.append(c)
        i += step
    return out

def _chunk_id(url, chunk_text_):
    h = hashlib.sha1((url + "::" + chunk_text_).encode("utf-8")).hexdigest()
    return f"{url}#sha1:{h}"

#PSE + DDG fallback not really using ddg for now
def google_pse_search(query, num=10):
    if not GOOGLE_API_KEY:
        raise RuntimeError("Missing GOOGLE_API_KEY in .env (Custom Search JSON API key).")
    
    key = ("google", query, num)
    if key in SEARCH_CACHE:
        return SEARCH_CACHE[key]

    url = "https://www.googleapis.com/customsearch/v1"
    params = {"q": query, "key": GOOGLE_API_KEY, "cx": GOOGLE_CSE_ID, "num": min(num, 10)}

    r = requests.get(url, params=params, timeout=20)

    if r.status_code == 403:
        raise RuntimeError("PSE 403: " + r.text)

    r.raise_for_status()
    data = r.json()

    out = []
    for item in data.get("items", []):
        link = item.get("link", "")
        out.append({
            "title": item.get("title", ""),
            "url": link,
            "snippet": item.get("snippet", ""),
            "domain": domain_of(link)
        })
    SEARCH_CACHE[key] = out
    return out

def ddg_search(query, num=10):
    key = ("ddg", query, num)
    if key in SEARCH_CACHE:
        return SEARCH_CACHE[key]

    url = "https://duckduckgo.com/html/"
    r = requests.post(url, data={"q": query}, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    html = r.text

    links = re.findall(r'class="result__a".*?href="([^"]+)"', html)
    out = []
    seen = set()
    for link in links:
        if link in seen:
            continue
        seen.add(link)
        out.append({"title": "", "url": link, "snippet": "", "domain": domain_of(link)})
        if len(out) >= num:
            break
    SEARCH_CACHE[key] = out
    return out


def build_search_queries(q):
    """Generate diverse search queries using LLM decomposition."""
    prompt = f"""
    Generate 3-5 diverse search queries for retrieving medical information about:
    "{q}"
    
    Rules:
    - Include medical terms, synonyms, and related concepts
    - Add terms like "statistics", "epidemiology", "prevalence"
    - Keep queries concise (5-10 words)
    - One query per line
    
    Queries:
    """
    
    try:
        resp = safe_generate_content(model=MODEL_LLM, contents=prompt)
        queries = [l.strip("-• ").strip() for l in resp.text.splitlines() if len(l.strip()) > 10]
        return [q] + queries[:4]  # Original + expansions
    except:
        return [q]
    


def fetch_and_extract(url):
    if url in FETCH_CACHE:
        return FETCH_CACHE[url]

    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            FETCH_CACHE[url] = None
            return None
        text = trafilatura.extract(downloaded, include_tables=True)
        if not text or len(text) < 300:
            FETCH_CACHE[url] = None
            return None

        FETCH_CACHE[url] = text
        return text
    except:
        FETCH_CACHE[url] = None
        return None

#gemini embeddings+local fallback using sentence transformers
def gemini_embed_many(texts, task_type):
    cfg = types.EmbedContentConfig(task_type=task_type, output_dimensionality=EMBED_DIM)
    res = safe_embed_content(model=MODEL_EMB, contents=texts, config=cfg)
    vecs = [e.values for e in res.embeddings]
    if EMBED_DIM != 3072:
        return [_norm(v) for v in vecs]
    return [list(v) for v in vecs]

def local_embed_many(texts):
    vecs = st_model.encode(texts, normalize_embeddings=True)
    return [v.astype(np.float32).tolist() for v in vecs]

def embed_many(texts, task_type):
    """
    Ensure embedding dimensionality matches the existing Chroma collection.
    If your collection was built with local 384-dim embeddings, keep using local
    to avoid dimension mismatch errors.
    """
    # If you're using all-MiniLM-L6-v2 (384 dims), force local embeddings always.
    #embedding dim mismatch bw local and gemini
    return local_embed_many(texts), "local"

def cosine_sim(a, b):
    a = np.array(a, dtype=np.float32)
    b = np.array(b, dtype=np.float32)
    return float(np.dot(a, b))

def upsert_chunks(url, title, chunks):
    ids = [_chunk_id(url, c) for c in chunks]

    existing = set()
    try:
        got = col.get(ids=ids, include=[])
        for _id in got.get("ids", []):
            existing.add(_id)
    except:
        pass

    new = [(i, c) for i, c in enumerate(chunks) if ids[i] not in existing]
    if not new:
        return {"stored": 0, "embed_backend": None}

    new = new[:MAX_NEW_CHUNKS_TO_EMBED_PER_RUN]
    new_ids = [_chunk_id(url, c) for _, c in new]
    new_docs = [c for _, c in new]
    metas = [{"url": url, "title": title, "domain": domain_of(url)} for _ in new_docs]

    stored = 0
    embed_backend = None

    for i in range(0, len(new_docs), BATCH_SIZE):
        batch_docs = new_docs[i:i+BATCH_SIZE]
        batch_ids  = new_ids[i:i+BATCH_SIZE]
        batch_meta = metas[i:i+BATCH_SIZE]

        embs, backend = embed_many(batch_docs, task_type="RETRIEVAL_DOCUMENT")
        embed_backend = backend

        col.upsert(ids=batch_ids, documents=batch_docs, embeddings=embs, metadatas=batch_meta)
        stored += len(batch_docs)

    return {"stored": stored, "embed_backend": embed_backend}








def retrieve_topk(query, k=TOPK):
    q_embs, backend = embed_many([query], task_type="RETRIEVAL_QUERY")
    q_emb = q_embs[0]

    res = col.query(query_embeddings=[q_emb], n_results=k, include=["documents", "metadatas", "embeddings"])
    docs = res["documents"][0] if res.get("documents") else []
    metas = res["metadatas"][0] if res.get("metadatas") else []
    embs = res["embeddings"][0] if res.get("embeddings") else []

    hits = []
    for doc, meta, emb in zip(docs, metas, embs):
        hits.append({"text": doc, "meta": meta, "sim": cosine_sim(q_emb, emb)})
    hits.sort(key=lambda x: x["sim"], reverse=True)

    return hits, backend





    TRUSTED_MEDICAL_DOMAINS = {
        "pubmed.ncbi.nlm.nih.gov": 10,
        "who.int": 10,
        "cdc.gov": 9,
        "nih.gov": 9,
        "thelancet.com": 8,
        "nejm.org": 8,
        "bmj.com": 8,
        "mayoclinic.org": 7,
        "medlineplus.gov": 7,
        # Add Pakistan-specific sources
        "nih.org.pk": 8,
        "nhsrc.pk": 7,
    }

    def score_source(url, base_similarity):
        """Boost similarity scores for trusted medical sources."""
        domain = domain_of(url)
        trust_boost = TRUSTED_MEDICAL_DOMAINS.get(domain, 1.0)
        return base_similarity * trust_boost

    for doc, meta, emb in zip(docs, metas, embs):
        base_sim = cosine_sim(q_emb, emb)
        final_sim = score_source(meta.get("url", ""), base_sim)
        hits.append({"text": doc, "meta": meta, "sim": final_sim})

    return hits, backend


# ---------------- TOKEN BUDGETING ---------------- #

# Rough but safe token estimator (~4 chars/token for English)
def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)

MAX_CONTEXT_TOKENS = 7000  # safe for Groq 70B (leave room for prompt + answer)


def pack_hits_by_token_budget(hits, max_tokens=MAX_CONTEXT_TOKENS):
    """
    Selects as many hits as fit under a token budget.
    Preserves similarity ordering.
    """
    packed = []
    used = 0

    for h in hits:
        t = estimate_tokens(h["text"])
        if used + t > max_tokens:
            break
        packed.append(h)
        used += t

    return packed, used



def llm_answer_or_context(query, hits):
    # ---- NEW: token-budgeted packing ----
    packed_hits, used_tokens = pack_hits_by_token_budget(hits)

    if not packed_hits:
        return "No retrieved context available yet.", "no_context"

    blocks = []
    for i, h in enumerate(packed_hits, 1):
        url = h["meta"].get("url", "")
        blocks.append(f"[{i}] {url}\n{h['text']}")

    context = "\n\n".join(blocks).strip()

    prompt = f"""
Answer using ONLY the context. Use citations like [1], [2].
If you cannot answer, say what is missing.

Question:
{query}

Context:
{context}
""".strip()

    try:
        resp = safe_generate_content(model=MODEL_LLM, contents=prompt)
        text = (resp.text or "").strip()
        if not text:
            return "Empty LLM response. Returning context:\n\n" + context, "empty_llm"
        return text, "llm"
    except ClientError as e:
        if _is_rate_limit(e):
            return "LLM rate/quota exhausted. Returning retrieved context:\n\n" + context, "no_llm"
        raise


#INGESTION PIPELINE
def web_ingest(query, debug=True):
    squeries = build_search_queries(query)
    if debug:
        print("\nSEARCH QUERIES:")
        for q in squeries:
            print("-", q)

    urls = []
    seen = set()

    for sq in squeries:
        results = []
        try:
            results = google_pse_search(sq, num=TOPN_WEB)
        except Exception as e:
            if debug:
                print("PSE error:", e)
            try:
                results = ddg_search(sq, num=TOPN_WEB)
                if debug:
                    print("Using DDG fallback for:", sq)
            except Exception as e2:
                if debug:
                    print("DDG error:", e2)
                results = []

        for r in results:
            u = r.get("url", "")
            if u and u not in seen:
                seen.add(u)
                urls.append(r)

    if debug:
        print("\nTOP URLS:")
        for r in urls[:TOPN_WEB]:
            print("-", r["url"])

    extracted_ok = 0
    total_stored = 0
    last_backend = None

    # for r in urls[:TOPN_WEB]:
    #     text = fetch_and_extract(r["url"])
    #     if not text:
    #         continue
    #     extracted_ok += 1

    #     chunks = chunk_text(text)
    #     stats = upsert_chunks(r["url"], r.get("title",""), chunks)
    #     total_stored += stats["stored"]
    #     last_backend = stats["embed_backend"]

    def _fetch_one(r):
        return r, fetch_and_extract(r["url"])

    with ThreadPoolExecutor(max_workers=6) as ex:
        futures = [ex.submit(_fetch_one, r) for r in urls[:TOPN_WEB]]

        for fut in as_completed(futures):
            r, text = fut.result()
            if not text:
                continue

            extracted_ok += 1
            chunks = chunk_text(text)[:MAX_CHUNKS_PER_PAGE]
            stats = upsert_chunks(r["url"], r.get("title",""), chunks)
            total_stored += stats["stored"]
            last_backend = stats["embed_backend"]


    if debug:
        print("\nEXTRACTION OK:", extracted_ok)
        print("STORED NEW CHUNKS:", total_stored)
        print("EMBED BACKEND USED FOR STORE:", last_backend)

    return {"search_queries": squeries, "results": urls[:TOPN_WEB], "chunks_added": total_stored}


import re

def should_skip_web(query: str) -> bool:
    """
    Returns True ONLY if the query is fundamentally non-public
    and should never be sent to web search or expanded.
    """

    q = query.lower()

    # Explicit patient-identifying signals
    patient_identifiers = [
        r"\bpatient\s+id\b",
        r"\bmrn\b",
        r"\bmedical\s+record\b",
        r"\bcase\s+file\b",
        r"\bclinical\s+note\b",
        r"\bdischarge\s+summary\b",
        r"\bimaging\s+(scan|report)\b",
        r"\bmri\b",
        r"\bct\s+scan\b",
        r"\bx[-\s]?ray\b",
        r"\becg\b",
        r"\behr\b",
    ]

    # Named individual pattern (very conservative)
    named_person = r"\b(patient|person|individual)\s+[A-Z][a-z]+\b"

    # Hospital-internal or proprietary phrasing
    internal_data = [
        r"\binternal\b",
        r"\bconfidential\b",
        r"\bprivate\b",
        r"\bunpublished\b",
        r"\bin-house\b",
    ]

    for pattern in patient_identifiers:
        if re.search(pattern, q):
            return True

    if re.search(named_person, query):
        return True

    for pattern in internal_data:
        if re.search(pattern, q):
            return True

    # Otherwise: let retrieval + expansion decide
    return False





# ---------------- QUERY EXPANSION ---------------- #   
def expand_query_regionally(query):
    """
    Broadens region progressively.
    Output is ordered from most specific -> broadest.
    """

    prompt = f"""
You are helping retrieve PUBLICLY AVAILABLE medical statistics.

Expand the following query by progressively broadening ONLY its
geographic scope (city → province/state → country → region → global).

Rules:
- Do NOT change disease, metric, or population
- Do NOT invent numbers
- Do NOT include explanations
- Each line must be a valid search query

Original query:
{query}

Expanded queries (one per line):
"""

    try:
        resp = safe_generate_content(model=MODEL_LLM, contents=prompt)
        lines = (resp.text or "").splitlines()
        expanded = [l.strip("-• ").strip() for l in lines if len(l.strip()) > 10]

        # Ensure original query is first
        out = [query]
        for q in expanded:
            if q.lower() != query.lower():
                out.append(q)

        return out[:4]   # hard cap for latency control

    except Exception:
        return [query]
    



def classify_query(query):
    """Classify query type to optimize retrieval strategy."""
    prompt = f"""
    Classify this medical query into ONE category:
    - "definition": asking what something is
    - "statistics": asking for numbers, rates, prevalence
    - "treatment": asking about protocols, medications
    - "regional": asking for location-specific data
    
    Query: "{query}"
    
    Category:
    """
    
    try:
        resp = safe_generate_content(model=MODEL_LLM, contents=prompt)
        category = resp.text.strip().lower()
        print("IT KINDA WORKING\n", category)
        return category if category in ["definition", "statistics", "treatment", "regional"] else "general"
    except:
        return "general"
    





def retrieval_is_promising(hits, min_hits=3, min_sim=0.35, min_diversity=2):
    """
    Enhanced retrieval quality check with multiple signals.
    """
    if len(hits) < min_hits:
        return False
    
    # Similarity check
    avg_sim = sum(h["sim"] for h in hits[:min_hits]) / min_hits
    if avg_sim < min_sim:
        return False
    
    # Diversity check (avoid single-source spam)
    domains = {h["meta"].get("domain", "") for h in hits[:5]}
    if len(domains) < min_diversity:
        return False
    
    # Recency check (prefer recent medical data)
    # You could parse dates from URLs/titles if available
    
    return True



def retrieval_is_reliable(hits):
    if not hits:
        return False

    # minimum number of chunks
    if len(hits) < 3:
        return False

    # similarity confidence
    avg_sim = sum(h["sim"] for h in hits[:3]) / 3
    if avg_sim < 0.35:
        return False

    # diversity check (avoid same-domain spam)
    domains = {h["url"].split("/")[2] for h in hits[:5]}
    if len(domains) < 2:
        return False

    return True


def llm_response_indicates_deferral(ans: str) -> bool:
    """
    Returns True if the LLM response indicates it is deferring due to lack of data
    at this geographic granularity.
    """
    ans_lower = ans.lower()

    deferral_phrases = [
        "not available",
        "no data for",
        "would require",
        "insufficient data",
        "not reported",
        "only available at",
        "no published statistics",
        "unknown at this level",
    ]

    # Also check if there are **any numbers**, if not, treat as deferral
    contains_number = bool(re.search(r"\d", ans_lower))

    if any(p in ans_lower for p in deferral_phrases):
        return True
    if not contains_number:
        return True

    return False

In [157]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

BRAVE_API_KEY = os.getenv("BRAVE_API_KEY")


def brave_search(query, num_results=10):
    """Search using Brave Search API and return list of website URLs."""
    
    if not BRAVE_API_KEY:
        raise RuntimeError("Missing BRAVE_API_KEY in .env file")
    
    url = "https://api.search.brave.com/res/v1/web/search"
    
    params = {
        "q": query,
        "count": min(num_results, 20)
    }
    
    headers = {
        "Accept": "application/json",
        "Accept-Encoding": "gzip",
        "X-Subscription-Token": BRAVE_API_KEY
    }
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=20)
        response.raise_for_status()
        data = response.json()
        
        results = data.get("web", {}).get("results", [])
        
        # Return full result objects with title, URL, and description
        search_results = []
        for item in results:
            search_results.append({
                "title": item.get("title", ""),
                "url": item.get("url", ""),
                "description": item.get("description", "")
            })
        
        return search_results
    
    except Exception as e:
        print(f"❌ Brave search error: {e}")
        return []

In [3]:
def build_context_from_results(search_results, max_results=10):
    """
    Build a context string from search results for LLM.

    Args:
        search_results: List of dicts with 'title', 'url', 'description'
        max_results: Maximum number of results to include

    Returns:
        Formatted context string
    """

    context_parts = []

    for i, result in enumerate(search_results[:max_results], 1):
        context_parts.append(
            f"[{i}] {result['title']}\n"
            f"URL: {result['url']}\n"
            f"Summary: {result['description']}\n"
        )

    context = "\n".join(context_parts)
    return context


def generate_answer_with_llm(query, context):
    """
    Generate answer using Groq LLM with search results as context.
    
    Args:
        query: User's question
        context: Search results formatted as context
    
    Returns:
        LLM-generated answer with citations
    """
    
    if not GROQ_API_KEY:
        raise RuntimeError("Missing GROQ_API_KEY in .env file")
    
    system_prompt = """You are a helpful medical information assistant. 
Answer the user's question using ONLY the provided search results context.

Rules:
- Use citations like [1], [2] to reference sources
- If the context doesn't contain enough information, say so
- Be accurate and cite your sources
- Focus on medical facts and statistics
- Keep answers clear and concise"""

    user_prompt = f"""Question: {query}

Search Results Context:
{context}

Please answer the question based on the above search results. Use citations [1], [2], etc. to reference specific sources."""

    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.2,
            max_tokens=1000
        )
        
        answer = response.choices[0].message.content.strip()
        return answer
    
    except Exception as e:
        return f"❌ Error generating answer: {e}"

In [4]:
def run_pipeline(query, debug=True):
    """
    Complete pipeline: Search -> Build Context -> Generate Answer
    
    Args:
        query: User's question
        debug: Print debug information
    
    Returns:
        Dict with 'query', 'search_results', 'context', 'answer'
    """
    
    query = query.strip()
    
    if debug:
        print("=" * 80)
        print(f"QUERY: {query}")
        print("=" * 80)
    
    # Step 1: Search with Brave
    if debug:
        print("\n🔍 Step 1: Searching with Brave...")
    
    search_results = brave_search(query, num_results=10)
    
    if debug:
        print(f"✓ Found {len(search_results)} results\n")
        print("BRAVE SEARCH RESULTS:")
        print("-" * 80)
        for i, result in enumerate(search_results, 1):
            print(f"{i}. {result['title']}")
            print(f"   {result['url']}")
            print(f"   {result['description'][:100]}...")
            print()
    
    if not search_results:
        return {
            "query": query,
            "search_results": [],
            "context": "",
            "answer": "No search results found. Unable to answer the question."
        }
    
    # Step 2: Build context from search results
    if debug:
        print("\n📝 Step 2: Building context for LLM...")
    
    context = build_context_from_results(search_results, max_results=10)
    
    if debug:
        print(f"✓ Context built ({len(context)} characters)\n")
    
    # Step 3: Generate answer with LLM
    if debug:
        print("\n🤖 Step 3: Generating answer with LLM...")
    
    answer = generate_answer_with_llm(query, context)
    
    if debug:
        print("✓ Answer generated\n")
        print("=" * 80)
        print("FINAL ANSWER:")
        print("=" * 80)
        print(answer)
        print("=" * 80)
    
    return {
        "query": query,
        "search_results": search_results,
        "context": context,
        "answer": answer
    }

In [ ]:
# run_pipeline("I want to know the number of people who have malaria in Punjab")

QUERY: I want to know the number of people who have malaria in Punjab

🔍 Step 1: Searching with Brave...
✓ Found 10 results

BRAVE SEARCH RESULTS:
--------------------------------------------------------------------------------
1. Occurrence and seasonal variation of human Plasmodium infection in Punjab Province, Pakistan - PMC
   https://pmc.ncbi.nlm.nih.gov/articles/PMC6836532/
   The 18S rRNA genes of Plasmodium species were amplified, sequenced, blast and the phylogenetic tree ...

2. Occurrence and seasonal variation of human Plasmodium infection in Punjab Province, Pakistan | BMC Infectious Diseases | Springer Nature Link
   https://link.springer.com/article/10.1186/s12879-019-4590-2
   The 18S rRNA genes of Plasmodium species were amplified, sequenced, blast and the phylogenetic tree ...

3. Surveillance-based estimation of the malaria disease burden in a low endemic state of Punjab, India, targeted for malaria elimination - PubMed
   https://pubmed.ncbi.nlm.nih.gov/33539517/
  

{'query': 'I want to know the number of people who have malaria in Punjab',
 'search_results': [{'title': 'Occurrence and seasonal variation of human Plasmodium infection in Punjab Province, Pakistan - PMC',
   'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6836532/',
   'description': 'The 18S rRNA genes of Plasmodium species were amplified, sequenced, blast and the phylogenetic tree was constructed based on sequences using online integrated tool MEGA7. The <strong>364 cases</strong> recruited from Northern Punjab with the highest incidence in Rawalpindi (25.5%) and lowest in Chakwal (15.9%).'},
  {'title': 'Occurrence and seasonal variation of human Plasmodium infection in Punjab Province, Pakistan | BMC Infectious Diseases | Springer Nature Link',
   'url': 'https://link.springer.com/article/10.1186/s12879-019-4590-2',
   'description': 'The 18S rRNA genes of Plasmodium species were amplified, sequenced, blast and the phylogenetic tree was constructed based on sequences using onli

In [156]:
run_pipeline('Which Angiotensin Receptor Blocker is most effective in reducing blood pressure in hypertensive Patients in Punjab in Pakistan?')

QUERY: Which Angiotensin Receptor Blocker is most effective in reducing blood pressure in hypertensive Patients in Punjab in Pakistan?

🔍 Step 1: Searching with Brave...
✓ Found 10 results

BRAVE SEARCH RESULTS:
--------------------------------------------------------------------------------
1. combination antihypertensive therapy: Topics by Science.gov
   https://www.science.gov/topicpages/c/combination+antihypertensive+therapy.html
   Clinical experience in treating hypertension with fixed-dose combination therapy: <strong>angiotensi...

2. Angiotensin II receptor blockers - Mayo Clinic
   https://www.mayoclinic.org/diseases-conditions/high-blood-pressure/in-depth/angiotensin-ii-receptor-blockers/art-20045009
   Kidney failure due to diabetes. ... Too much potassium in the blood. Swelling of the skin due to ext...

3. Blood pressure lowering efficacy of angiotensin receptor blockers for primary hypertension - PMC
   https://pmc.ncbi.nlm.nih.gov/articles/PMC6669255/
   Participants we

{'query': 'Which Angiotensin Receptor Blocker is most effective in reducing blood pressure in hypertensive Patients in Punjab in Pakistan?',
 'search_results': [{'title': 'combination antihypertensive therapy: Topics by Science.gov',
   'url': 'https://www.science.gov/topicpages/c/combination+antihypertensive+therapy.html',
   'description': 'Clinical experience in treating hypertension with fixed-dose combination therapy: <strong>angiotensin II receptor blocker losartan plus hydrochlorothiazide</strong>. ... The goal of antihypertensive treatment is to reduce cardiovascular and cerebrovascular events associated with high blood pressure.'},
  {'title': 'Angiotensin II receptor blockers - Mayo Clinic',
   'url': 'https://www.mayoclinic.org/diseases-conditions/high-blood-pressure/in-depth/angiotensin-ii-receptor-blockers/art-20045009',
   'description': 'Kidney failure due to diabetes. ... Too much potassium in the blood. Swelling of the skin due to extra fluid. Some people taking the <s

In [5]:
def run_pipeline2(query, debug=True):
    query = query.strip()
    query_type = classify_query(query)
    
    if query_type == "definition":
        # Skip web search for well-known medical terms
        # Increase reliance on local vector DB
        pass
    elif query_type == "regional":
        # Immediately trigger regional expansion
        pass

    if debug:
        print("\nORIGINAL QUERY:", query)

    # ---------- FIRST PASS: LOCAL VECTOR SEARCH ----------
    hits, emb_backend = retrieve_topk(query)

    ans = ""
    llm_mode = "no_llm"

    if retrieval_is_promising(hits):   # 🔑 gate before LLM
        ans, llm_mode = llm_answer_or_context(query, hits)

    weak = (
        len(hits) == 0 or
        llm_response_indicates_deferral(ans) or
        len(ans) < 120
    )

    if not weak:
        return {
            "mode": "local",
            "answer": ans,
            "hits": hits,
            "retrieval_backend": emb_backend,
            "llm_mode": llm_mode,
            "query_used": query,
            "expanded": False
        }

    # ---------- PRIVACY GATE ----------
    if should_skip_web(query):
        return {
            "mode": "blocked",
            "answer": "Insufficient public data.",
            "expanded": False,
            "reason": "Query involves private or non-public information",
            "query_used": query
        }

    # ---------- QUERY EXPANSION ----------
    expanded_queries = expand_query_regionally(query)

    if debug:
        print("\nQUERY EXPANSIONS:")
        for i, q in enumerate(expanded_queries):
            print(f"[{i}] {q}")

    # ---------- EXPANDED WEB INGEST ----------
    for level, q_exp in enumerate(expanded_queries[1:], start=1):

        if debug:
            print(f"\n--- Expansion Level {level} ---")
            print("Query:", q_exp)

        web_ingest(q_exp, debug=debug)

        hits_exp, emb_backend_exp = retrieve_topk(q_exp)

        if not retrieval_is_promising(hits_exp):
            continue

        ans_exp, llm_mode_exp = llm_answer_or_context(q_exp, hits_exp)

        weak_exp = (
            len(hits_exp) == 0 or
            llm_response_indicates_deferral(ans_exp) or
            len(ans_exp) < 120
        )

        if not weak_exp:
            return {
                "mode": "expanded",
                "answer": ans_exp,
                "hits": hits_exp,
                "retrieval_backend": emb_backend_exp,
                "llm_mode": llm_mode_exp,
                "query_used": q_exp,
                "expanded": True,
                "expansion_level": level
            }

    # ---------- NOTHING WORKED ----------
    return {
        "mode": "expanded_but_insufficient",
        "answer": "Even after broadening the geographic scope, no reliable published data was found.",
        "hits": [],
        "retrieval_backend": None,
        "llm_mode": "no_data",
        "query_used": expanded_queries[-1],
        "expanded": True
    }


In [ ]:
out = run_pipeline2("Incidence of cardiac arrest in UK", debug=True)
print("\nMODE:", out["mode"])
print("RETRIEVAL_BACKEND:", out.get("retrieval_backend"))
print("LLM_MODE:", out.get("llm_mode"))
print("\nANSWER:\n", out["answer"])

In [6]:
url = "https://pmc.ncbi.nlm.nih.gov/articles/PMC11415798/"

downloaded = trafilatura.fetch_url(url)

if not downloaded:
    print("❌ Failed to download page")
else:
    text = trafilatura.extract(downloaded, include_tables=True)
    print("===== TRAFILATURA OUTPUT START =====\n")
    print(text)
    print("\n===== TRAFILATURA OUTPUT END =====")


===== TRAFILATURA OUTPUT START =====

Abstract
Background
In hospital cardiac arrest is associated with poor survival despite basic and advanced life support measures. This study aimed to identify the clinical characteristics and outcomes of cardiac arrests occurring during in-hospital admission to the tertiary care center in Pakistan.
Method
A retrospective, cross-sectional study at Aga Khan University Hospital from 2021 to 2023 analyzed 230 cardiac arrest cases. Data included demographics, arrest type, timing, initial rhythm, resuscitation duration, and arrest location. American Heart Association guidelines were adhered to for life support. The main outcomes focused on the return of spontaneous circulation survival to hospital discharge.
Results
During the study, 230 cardiac arrests were observed: 152 in adults (mean age 57.8, 142 shockable cases, ROSC 52.6 %, alive at discharge 28.3 %) and 78 in pediatric patients (mean age 4.99, non-shockable rhythm 85.9 %, ROSC 51.3 %, alive at di

In [7]:
from docx import Document
import re

doc = Document("Queries.docx")

queries = []
current_query = ""

def clean_text(text):
    # Remove weird control characters (Word form feeds etc.)
    text = text.replace("\f", "").strip()
    text = re.sub(r"\s+", " ", text)
    
    return text

for para in doc.paragraphs:
    text = clean_text(para.text)

    if not text:
        continue

    # Ignore section labels / names (single words, no punctuation)
    if (
        len(text.split()) <= 2
        and not text.endswith("?")
        and text.isalpha()
    ):
        continue

    # If line ends with '?', it's the END of a query
    if text.endswith("?"):
        if current_query:
            full_query = f"{current_query} {text}".strip()
            queries.append(full_query)
            current_query = ""
        else:
            queries.append(text)

    else:
        # Line is part of a wrapped/multi-line query
        current_query += " " + text if current_query else text

print(f"Extracted {len(queries)} queries")

Extracted 104 queries


In [8]:
for i in range(len(queries)):
    print(queries[i])

What percentage of patients presenting with Diabetic Ketoacidosis in tertiary care hospitals in Lahore go into DKA because of inability to afford insulin shots?
What percentage of patients are referred to ophthalmology for screening for diabetic retinopathy from a diabetes clinic in Ganga Ram Hospital in Lahore?
What percentage of breast cancer patients undergoing surgery for breast cancer at Mayo Hospital in Lahore have tumors that are negative for estrogen, progesterone and HER-2 Neu receptors ( triple negative disease)?
Which Angiotensin Receptor Blocker is most effective in reducing blood pressure in hypertensive Patients in Punjab in Pakistan?
What is the difference in no-show rates for chemotherapy treatments between men and women undergoing cancer treatment in Shaukat Khanum Hospital in Lahore?
What is the difference between the no-show rates for chemotherapy treatments in early vs late stages of cancer progression in Lahore? Also bifurcate this data by gender. What percentage o

In [9]:
def dedupe_queries(queries):
    seen = set()
    out = []
    for q in queries:
        qn = q.lower().strip()
        if qn not in seen:
            seen.add(qn)
            out.append(q)
    return out

queries = dedupe_queries(queries)
print(f"After dedupe: {len(queries)} queries")

After dedupe: 104 queries


In [10]:
print(run_pipeline2(queries[32], debug=True))

IT KINDA WORKING
 the category for this medical query is: "statistics" [1] and also "regional" [2]. however, since the instruction is to classify into one category, the best fit would be "statistics" as the query is primarily asking for numbers and rates. 

category: "statistics"

ORIGINAL QUERY: What percentage of patient’s partners are motivated to get treated for an STD requiring partner treatment in tertiary care hospitals of Punjab? Also compare with that of KPK. What is the incidence of wheat pill poisoning (zinc or aluminium phosphide) cases presenting at Lahore General Hospital, Lahore?

QUERY EXPANSIONS:
[0] What percentage of patient’s partners are motivated to get treated for an STD requiring partner treatment in tertiary care hospitals of Punjab? Also compare with that of KPK. What is the incidence of wheat pill poisoning (zinc or aluminium phosphide) cases presenting at Lahore General Hospital, Lahore?
[1] What percentage of patient’s partners are motivated to get treated 

In [165]:
for i in range(46, 57):
    out = run_pipeline(queries[i], debug=False)
    print('='*80)
    print("QUERY:\n ", out['query'])

    print("\nANSWER:\n ", out['answer'])
    print('='*80)
    print('\n')

QUERY:
  How frequently are autoclaves in the gynecological operation theatres of Lady Willingdon Hospital inspected and maintained according to established protocols?



ANSWER:
  The frequency of inspection and maintenance of autoclaves in the gynecological operation theatres of Lady Willingdon Hospital is not explicitly stated in the provided search results context. However, according to general guidelines for central sterile supply departments (CSSD), autoclaves should be regularly inspected and maintained to ensure proper function and sterility [2]. Additionally, the Bowie-Dick test should be performed to verify the autoclave's effectiveness, and the sterilizer should not be used until it passes the test [9]. Unfortunately, the context does not provide specific information about the protocols followed at Lady Willingdon Hospital. [1], [6], and [8] provide general information about autoclave maintenance and sterile processing, but do not mention the hospital specifically. Therefore